# 2. 예비 지식: PBRFT에서 에이전트 RL로의 전환

이 노트북은 선호도 기반 강화 학습 미세 조정(PBRFT)에서 에이전트 강화 학습(Agentic RL)으로의 패러다임 전환을 소개한다.

단일 단계의 단순한 설정에서 벗어나 긴 호흡의 부분 관찰 가능한 환경으로 이동함으로써, 훈련 역학이 어떻게 변화하고 계획 수립, 도구 사용, 기억이 어떻게 나타나는지 이해하게 될 것이다.

**핵심 개념**: 
- PBRFT = 단일 단계 MDP, 텍스트만 생성, 즉각 보상
- Agentic RL = 다단계 POMDP, 텍스트 + 도구, 누적 보상
- Generative Agent Memory = Perceive → Memory → Retrieve → Act의 순환 구조

In [2]:
# 설정: 필요한 라이브러리 임포트
from utils_openai import *
import numpy as np
from datetime import datetime

print("OpenAI API 유틸리티 로드 완료했다.")
print("주요 함수: ask(), chat(), tool_schema(), run_tools(), llm_reward()")
print("주요 클래스: MemoryStream")

OpenAI API 유틸리티 로드 완료했다.
주요 함수: ask(), chat(), tool_schema(), run_tools(), llm_reward()
주요 클래스: MemoryStream


## 실습 1: PBRFT 스타일의 단일 턴 상호작용

**목표**: 단일 프롬프트를 입력하고 한 번의 응답을 받아 점수를 매기기

In [3]:
# 1단계: PBRFT 스타일의 단순 호출
prompt = "프랑스의 수도는 어디인가?"

# ask() 함수로 즉각 응답 생성 (PBRFT 방식)
response = ask(
    prompt=prompt,
    system="당신은 유용한 AI 어시스턴트다.",
    temperature=0.0  # 결정론적 응답
)

heading("PBRFT 단일 턴")
step_print(1, "입력", prompt)
step_print(2, "응답", response)

# 2단계: 응답 평가 (종료 신호)
score = len(response.split()) * 0.5  # 간단한 휴리스틱
step_print(3, "점수", f"{score:.2f} (응답이 완전함)")

print("PBRFT는 이것으로 완료다. 더 이상 상호작용이 없다.")


────────────────────────────────────────
  PBRFT 단일 턴
────────────────────────────────────────
  [Step 1] 입력
    프랑스의 수도는 어디인가?
  [Step 2] 응답
    프랑스의 수도는 파리입니다.
  [Step 3] 점수
    1.50 (응답이 완전함)
PBRFT는 이것으로 완료다. 더 이상 상호작용이 없다.


## 실습 2: Agentic RL 스타일의 다단계 상호작용

**목표**: 메모리를 사용하며 여러 단계에 걸쳐 문제를 해결하기

In [4]:
# 1단계: 메모리 스트림 초기화
memory = MemoryStream()
initial_query = "주어진 숫자들의 합을 구하고 설명해주세요: 1부터 10까지"

# 2단계: Perceive - 초기 관찰을 메모리에 저장
memory.add(initial_query, importance=0.8)
step_print(1, "Perceive", f"과제 인식: {initial_query}")

messages = [
    {
        "role": "system",
        "content": "당신은 문제 해결 에이전트다. 각 단계마다 생각하고 진행 상황을 설명하세요."
    },
    {"role": "user", "content": initial_query}
]

# 3단계: Plan & Act - 다단계 상호작용
for step in range(2):
    # Retrieve: 관련 기억 검색
    retrieved = memory.retrieve(initial_query, top_k=10)
    step_print(2 + step*2, "Retrieve", f"{len(retrieved)}개의 기억을 검색했다")
    
    # Act: 행동 생성
    response_obj = chat(
        messages=messages,
        temperature=0.5,
        max_tokens=200
    )
    
    # [중요] 응답 객체에서 텍스트 내용 추출
    response_text = response_obj.choices[0].message.content
    
    step_print(3 + step*2, "Act", f"{response_text}...")
    
    # Memory: 메모리 업데이트 (텍스트로 저장)
    memory.add(response_text, importance=0.7)
    messages.append({"role": "assistant", "content": response_text})
    
    # 종료 확인
    if "55" in response_text or "완료" in response_text:
        step_print(4 + step*2, "Status", "과제 완료했다")
        break
    
    # 다음 턴을 위한 프롬프트
    messages.append({"role": "user", "content": "계속 진행하세요"})

print("Agentic RL은 메모리와 다단계 상호작용으로 복잡한 문제를 해결했다.")

  [Step 1] Perceive
    과제 인식: 주어진 숫자들의 합을 구하고 설명해주세요: 1부터 10까지
  [Step 2] Retrieve
    1개의 기억을 검색했다
  [Step 3] Act
    1부터 10까지의 숫자들의 합을 구하는 방법은 여러 가지가 있지만, 가장 간단한 방법은 숫자를 하나씩 더하는 것입니다. 
    
    1. **숫자 리스트 작성**: 1, 2, 3, 4, 5, 6, 7, 8, 9, 10
    2. **합산 과정**: 각 숫자를 순차적으로 더합니다.
       - 1 + 2 = 3
       - 3 + 3 = 6
       - 6 + 4 = 10
       - 10 + 5 = 15
       - 15 + 6 = 21
       - 21 + 7 = 28
       - 28 + 8 = 36
       - 36 + 9 = 45
       - 45 + 10 = 55
    
    따라서, 1부터 10...
  [Step 4] Status
    과제 완료했다
Agentic RL은 메모리와 다단계 상호작용으로 복잡한 문제를 해결했다.


## 핵심 비교

| 측면 | PBRFT | Agentic RL |
|------|-------|----------|
| **호출 함수** | `ask()` | `chat()`, `MemoryStream` |
| **단계 수** | 1 | 여러 개 |
| **메모리** | 없음 | 있음 (MemoryStream) |
| **보상** | 최종 점수 | 다단계 신호 |
| **복잡도** | 낮음 | 높음 |
| **능력** | 응답 생성 | 계획, 도구, 추론 |

## 실습 3: 할인 누적 보상 이해하기

**목표**: 왜 Agentic RL이 할인율(γ)을 사용하는지 이해하기

In [5]:
# 스텝별 보상 시뮬레이션
rewards = [1.0, 2.0, 3.0, 20.0]  # 마지막 단계에서 큰 보상
gamma = 0.9

heading("할인 누적 보상 계산")

# 비할인 합계
sum_reward = sum(rewards)
kv_print([("비할인 합계", f"{sum_reward:.1f}")])

# 할인 적용
discounted = sum(r * (gamma ** i) for i, r in enumerate(rewards))
kv_print([("할인 누적 (γ=0.9)", f"{discounted:.2f}")])

# 미래 보상의 영향 시각화
print("단계별 할인율:")
for i, r in enumerate(rewards):
    discounted_r = r * (gamma ** i)
    print(f"  Step {i}: 보상 {r:.1f} × γ^{i} = {discounted_r:.3f} (원본의 {(gamma**i)*100:.1f}%)")

print("해석: 미래 보상도 중요하지만, 현재에 더 가까운 보상이 더 중요하다.")


────────────────────────────────────────
  할인 누적 보상 계산
────────────────────────────────────────
  비할인 합계  26.0
  할인 누적 (γ=0.9)  19.81
단계별 할인율:
  Step 0: 보상 1.0 × γ^0 = 1.000 (원본의 100.0%)
  Step 1: 보상 2.0 × γ^1 = 1.800 (원본의 90.0%)
  Step 2: 보상 3.0 × γ^2 = 2.430 (원본의 81.0%)
  Step 3: 보상 20.0 × γ^3 = 14.580 (원본의 72.9%)
해석: 미래 보상도 중요하지만, 현재에 더 가까운 보상이 더 중요하다.


## 요약

이 노트북에서는 다음을 배웠다:

1. **PBRFT**: `ask()`로 단일 응답 생성 → 즉각 보상
2. **Agentic RL**: `chat()`과 `MemoryStream`으로 다단계 상호작용 → 누적 보상
3. **메모리의 역할**: Perceive-Memory-Retrieve-Act 순환으로 장기 추론 가능
4. **할인율**: 미래 보상도 고려하되, 지수적으로 감쇠